In [1]:
import pandas as pd
import plotnine as p9
import numpy as np
import os

In [2]:
PLOT_DIR = 'data/general_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

In [3]:
pdmorl_file = 'data/pdmorl_stats.csv'
pcpl_file = 'data/pcpl_stats.csv'

In [4]:
pdmorl = pd.read_csv(pdmorl_file)
pdmorl

,problem,config,checkpoint,hypervolume,normalized_hv,percent_non_dominated,ordering_score,pref_grid_step,tau,learning_rate,gamma,batch_size,seed,timestamp
0,problem_0,problem_0.yml,NaN,437.393187,0.975834,1.000000,1.000000,0.05,0.050,0.0001,0.95,64,2,2026-04-24T18:05:50
1,problem_0,problem_0.yml,NaN,444.785398,0.992326,0.714286,1.000000,0.05,0.050,0.0001,0.95,64,3,2026-04-24T18:05:53
2,problem_0,problem_0.yml,NaN,444.785398,0.992326,1.000000,1.000000,0.05,0.050,0.0001,0.95,64,1,2026-04-24T18:05:53
3,problem_0,problem_0.yml,NaN,437.079403,0.975134,0.523810,1.000000,0.05,0.050,0.0001,0.95,64,0,2026-04-24T18:05:57
4,problem_0,problem_0.yml,NaN,444.785398,0.992326,0.619048,1.000000,0.05,0.050,0.0001,0.95,64,4,2026-04-24T18:36:57
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,problem_5e,problem_5e.yml,NaN,278809.273724,0.614014,0.205910,0.618246,0.05,0.005,0.0003,0.99,256,0,2026-04-25T10:44:53
76,problem_5e,problem_5e.yml,NaN,284555.303028,0.626669,0.676172,0.461971,0.05,0.005,0.0003,0.99,256,1,2026-04-25T10:54:10
77,problem_5e,problem_5e.yml,NaN,277291.924635,0.610673,0.267928,0.628444,0.05,0.005,0.0003,0.99,256,2,2026-04-25T11:10:32
78,problem_5e,problem_5e.yml,NaN,304007.590694,0.669508,0.373047,0.594596,0.05,0.005,0.0003,0.99,256,3,2026-04-25T11:15:15


In [5]:
pcpl = pd.read_csv(pcpl_file)
pcpl

,env_name,hypervolume,normalized_hv,non_dominated,ordering,architecture_idx,clip_range,entropy,lr,n_epochs,n_steps,scalarization_method,seed,smoothness,target_kl,sweep_timesteps,wandb_run_name
0,problem_0,440.922607,0.983708,1.000000,1.000000,0,0.226605,0.005462,0.000053,20,2048,smooth_tchebycheff,1,0.019041,0.013451,200000,morning-sweep-28
1,problem_0,448.225042,1.000000,1.000000,1.000000,0,0.226605,0.005462,0.000053,20,2048,smooth_tchebycheff,2,0.019041,0.013451,200000,morning-sweep-28
2,problem_0,448.225042,1.000000,1.000000,1.000000,0,0.226605,0.005462,0.000053,20,2048,smooth_tchebycheff,3,0.019041,0.013451,200000,morning-sweep-28
3,problem_0,448.225042,1.000000,1.000000,1.000000,0,0.226605,0.005462,0.000053,20,2048,smooth_tchebycheff,4,0.019041,0.013451,200000,morning-sweep-28
4,problem_0,448.225042,1.000000,1.000000,1.000000,0,0.226605,0.005462,0.000053,20,2048,smooth_tchebycheff,0,0.019041,0.013451,200000,morning-sweep-28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,problem_5e,342527.817214,0.754340,0.218681,0.762760,0,0.192599,0.062361,0.000099,20,2048,smooth_tchebycheff,0,0.064093,0.016039,200000,dazzling-sweep-24
76,problem_5e,338847.369605,0.746235,0.231868,0.608435,0,0.192599,0.062361,0.000099,20,2048,smooth_tchebycheff,1,0.064093,0.016039,200000,dazzling-sweep-24
77,problem_5e,342271.693519,0.753776,0.234615,0.781541,0,0.192599,0.062361,0.000099,20,2048,smooth_tchebycheff,2,0.064093,0.016039,200000,dazzling-sweep-24
78,problem_5e,267530.002666,0.589174,0.412088,0.920714,0,0.192599,0.062361,0.000099,20,2048,smooth_tchebycheff,3,0.064093,0.016039,200000,dazzling-sweep-24


In [6]:
def to_plot_df(
        df, 
        problem_col = 'problem', 
        metric_map = {
            'normalized_hv': 'HV Ratio',
            'percent_non_dominated': '% Non-Dominated',
            'ordering_score': 'Ordering Score',
        },
        run_idx = 'seed',
):
    """Extract, tidy, and pretify names."""
    plot_df = ( 
        df[[problem_col] + [k for k in metric_map.keys()] + [run_idx]]
        .set_index([problem_col, run_idx])
        .rename_axis('metric',axis=1)
        .stack()
    ).rename('score').reset_index()
    plot_df['metric'] = pd.Categorical(
        values=plot_df['metric'].map(metric_map),
        categories=[v for _,v in metric_map.items()],
        ordered=True
    )
    plot_df.rename(columns={problem_col:'Problem'}, inplace=True)
    plot_df['Problem'] = plot_df['Problem'].map({n:n.split('_')[1] for n in plot_df['Problem'].unique()})
    return plot_df

In [7]:
pdmorl_plot = to_plot_df(pdmorl)
display(pdmorl_plot.head(3))
pdmorl_plot.groupby(by='metric', observed=True)['score'].describe()

,Problem,seed,metric,score
0,0,2,HV Ratio,0.975834
1,0,2,% Non-Dominated,1.000000
2,0,2,Ordering Score,1.000000


,count,mean,std,min,25%,50%,75%,max
metric,,,,,,,,
HV Ratio,80.0,0.562444,0.290352,0.000407,0.366536,0.604840,0.752157,0.992326
% Non-Dominated,80.0,0.300245,0.335205,0.000188,0.028444,0.180642,0.571429,1.000000
Ordering Score,80.0,0.844984,0.152956,0.434120,0.740010,0.877209,1.000000,1.000000


In [8]:
pcpl_plot = to_plot_df(
    pcpl,
    problem_col='env_name',
    metric_map = {
        'normalized_hv': 'HV Ratio',
        'non_dominated': '% Non-Dominated',
        'ordering': 'Ordering Score',
    },    
    run_idx = 'seed',
)

In [9]:
display(pcpl_plot.head(3))
pcpl_plot.groupby(by=['metric'], observed=True)['score'].describe()

,Problem,seed,metric,score
0,0,1,HV Ratio,0.983708
1,0,1,% Non-Dominated,1.000000
2,0,1,Ordering Score,1.000000


,count,mean,std,min,25%,50%,75%,max
metric,,,,,,,,
HV Ratio,80.0,0.717013,0.232639,0.064898,0.587900,0.741135,0.892740,1.0
% Non-Dominated,80.0,0.453420,0.364621,0.001099,0.148352,0.307692,0.770055,1.0
Ordering Score,80.0,0.911822,0.107917,0.545746,0.877789,0.951027,0.986667,1.0


In [10]:
plot_df = pd.concat([pdmorl_plot, pcpl_plot], keys=['PD-MORL', 'PCPL-PPO'], names=['Architecture'])
plot_df['Subproblem'] = plot_df['Problem'].apply(lambda s: s[1:])
plot_df['Problem Set'] = plot_df['Problem'].apply(lambda s: s[0])
plot_df

Problem  seed           metric     score Subproblem  \
Architecture                                                           
PD-MORL      0         0     2         HV Ratio  0.975834              
             1         0     2  % Non-Dominated  1.000000              
             2         0     2   Ordering Score  1.000000              
             3         0     3         HV Ratio  0.992326              
             4         0     3  % Non-Dominated  0.714286              
...                  ...   ...              ...       ...        ...   
PCPL-PPO     235      5e     3  % Non-Dominated  0.412088          e   
             236      5e     3   Ordering Score  0.920714          e   
             237      5e     4         HV Ratio  0.760383          e   
             238      5e     4  % Non-Dominated  0.342857          e   
             239      5e     4   Ordering Score  0.767682          e   

                 Problem Set  
Architecture                  
PD-MORL      0             0  
             1             0  
             2             0  
             3             0  
             4             0  
...                      ...  
PCPL-PPO     235           5  
             236           5  
             237           5  
             238           5  
             239           5  

[480 rows x 6 columns]

In [ ]:
p = (
    p9.ggplot(data=plot_df.reset_index(), mapping=p9.aes(x='Problem', y='score', fill='Architecture', color='Architecture'))
    + p9.facet_grid('metric~.')
    + p9.stat_summary(
        geom='point', 
        fun_y=np.mean, 
        size=3,
        shape='_',
        stroke=1,
    )
    + p9.geom_point(
        position=p9.position_dodge(width=0.7),
        color='none',
        alpha=0.3,
        size=1.0,
    )
    + p9.theme_classic()
    + p9.theme(
        figure_size=(6.5,4.5), 
        panel_grid_major_y=p9.element_line(color='lightgray', alpha=0.5),
        legend_direction='horizontal', 
        legend_position='top',
        legend_box_margin=0,
        legend_box_spacing=0,
        strip_background=p9.element_blank(),    )
    + p9.scale_color_brewer(type='qual', palette=2)
    + p9.scale_fill_brewer(type='qual', palette=2)
    + p9.ylab('Score')
)

p.save(f'{PLOT_DIR}/problems1-5_v3.pdf')
display(p)

In [ ]:
p = (
    p9.ggplot(data=plot_df.reset_index(), mapping=p9.aes(x='Problem', y='score', fill='Architecture', color='Architecture'))
    + p9.facet_grid('metric~.')
    + p9.stat_summary(
        geom='point', 
        fun_y=np.mean, 
        size=1.0,
        shape='d',
        stroke=1,
    )
    + p9.geom_point(
        position=p9.position_dodge(width=0.7),
        color='none',
        alpha=0.3,
        size=1.0,
    )
    + p9.theme_classic()
    + p9.theme(
        figure_size=(6.5,4.5), 
        panel_grid_major_y=p9.element_line(color='lightgray', alpha=0.5),
        legend_direction='horizontal', 
        legend_position='top',
        legend_box_margin=0,
        legend_box_spacing=0,
        strip_background=p9.element_blank(),    )
    + p9.scale_color_brewer(type='qual', palette=2)
    + p9.scale_fill_brewer(type='qual', palette=2)
    + p9.ylab('Score')
)
p
p.save(f'{PLOT_DIR}/problems1-5_v4.pdf')
display(p)